# 01 — Dataset Creation & Raw Versioning

Bu notebook ana tez soru havuzunun oluşturulması ve **immutable raw dataset snapshot'larının**
üretilmesi için kullanılır.

## Temel prensip

- `questions_master.json` yaşayan / sürekli büyüyen soru havuzudur.
- Başarılı her batch commit'inden sonra master'ın yeni hali değişmez bir `raw_vXXX.json`
  snapshot'ına dönüştürülür.
- Eski `raw_vXXX.json` dosyalarının üzerine **asla yazılmaz**.
- Sonraki deney notebook'ları doğrudan `questions_master.json` okumaz;
  seçilmiş bir raw/validated versiyonu okur.
- Mevcut eski master verisi korunur. Versioning sistemine geçiş sırasında master JSON'a
  yazma yapılmaz.

Bu notebook **LLM inference yapmaz**.

## 1. Google Drive'ı bağla ve proje klasörlerini oluştur

Önerilen Drive yapısı:

```text
LLM_Confidence_Thesis/
├── data/
│   ├── raw/
│   │   ├── questions_master.json
│   │   └── versions/
│   └── validated/
├── notebooks/
├── runs/
├── features/
├── models/
└── results/
```

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/LLM_Confidence_Thesis")

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"

# Eski 01 notebook'unun pre-commit snapshot klasörü.
LEGACY_VERSIONS_DIR = DATA_RAW_DIR / "versions"

# Yeni immutable raw dataset sürümleri.
RAW_SNAPSHOTS_DIR = DATA_RAW_DIR / "snapshots"
RAW_MANIFESTS_DIR = RAW_SNAPSHOTS_DIR / "manifests"

# Commit öncesi geri dönüş amaçlı güvenlik yedekleri.
SAFETY_BACKUPS_DIR = DATA_RAW_DIR / "safety_backups"

DATA_VALIDATED_DIR = PROJECT_ROOT / "data" / "validated"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
RUNS_DIR = PROJECT_ROOT / "runs"
FEATURES_DIR = PROJECT_ROOT / "features"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

for directory in [
    DATA_RAW_DIR,
    LEGACY_VERSIONS_DIR,
    RAW_SNAPSHOTS_DIR,
    RAW_MANIFESTS_DIR,
    SAFETY_BACKUPS_DIR,
    DATA_VALIDATED_DIR,
    NOTEBOOKS_DIR,
    RUNS_DIR,
    FEATURES_DIR,
    MODELS_DIR,
    RESULTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

MASTER_JSON_PATH = DATA_RAW_DIR / "questions_master.json"

print("Project root:", PROJECT_ROOT)
print("Living master:", MASTER_JSON_PATH)
print("Raw snapshots:", RAW_SNAPSHOTS_DIR)
print("Safety backups:", SAFETY_BACKUPS_DIR)

## 2. Veri şeması

Her soru aşağıdaki alanlara sahip olacaktır:

- `question_id`: otomatik üretilen benzersiz kimlik
- `question`: Türkçe açık uçlu olgusal soru
- `reference_answer`: canonical doğru cevap
- `accepted_aliases`: doğru kabul edilecek alternatif cevaplar
- `answer_type`: cevabın yapısal türü
- `category`: konu alanı
- `source_type`: kaynağın türü (`generated`, `dataset`, `manual`, `other`)
- `source_name`: varsa kaynak/dataset adı
- `source_reference`: varsa URL, doküman veya kayıt kimliği
- `verification_status`: `unverified`, `verified`, `rejected`
- `notes`: editöryal not
- `batch_id`: ekleme batch'i
- `created_at_utc`: eklenme zamanı

`verification_status` özellikle önemlidir. Ana deney veri setine geçmeden önce ground-truth cevaplar doğrulanacaktır.

In [ ]:
ANSWER_TYPES = {
    "ENTITY_PERSON",
    "ENTITY_LOCATION",
    "ENTITY_ORGANIZATION",
    "ENTITY_OTHER",
    "DATE",
    "NUMBER",
    "NUMBER_WITH_UNIT",
    "SHORT_PHRASE",
    "BINARY",
}

SOURCE_TYPES = {"generated", "dataset", "manual", "other"}
VERIFICATION_STATUSES = {"unverified", "verified", "rejected"}

REQUIRED_INPUT_FIELDS = {
    "question",
    "reference_answer",
    "accepted_aliases",
    "answer_type",
    "category",
}

OPTIONAL_INPUT_DEFAULTS = {
    "source_type": "generated",
    "source_name": "",
    "source_reference": "",
    "verification_status": "unverified",
    "notes": "",
}

## 3. Normalization ve duplicate kontrol yardımcıları

Bu normalization yalnızca veri yönetimi ve duplicate tespiti içindir. Model cevaplarını skorlamak için kullanılacak normalization daha sonra ayrı notebook'ta dondurulacaktır.

In [ ]:
import re
import unicodedata


def normalize_for_duplicate_check(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFKC", text)
    text = text.replace("İ", "i").replace("I", "ı")
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_alias_for_duplicate_check(text: str) -> str:
    return normalize_for_duplicate_check(text)

## 4. Master JSON'ı oluştur / yükle

Master dosyanın kökü bir JSON listesi olacaktır.

In [ ]:
import json


def load_master_dataset():
    if not MASTER_JSON_PATH.exists():
        return []

    with open(MASTER_JSON_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError("Master JSON kök yapısı list olmalıdır.")

    return data


def save_master_dataset(data):
    with open(MASTER_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


master_data = load_master_dataset()
if not MASTER_JSON_PATH.exists():
    save_master_dataset(master_data)

print("Mevcut soru sayısı:", len(master_data))

## 4B. Mevcut master verisini güvenli biçimde versioning sistemine taşı

Bu bölüm **mevcut `questions_master.json` dosyasını değiştirmez**.

İlk çalıştırmada:

1. Mevcut master'ın birebir güvenlik yedeğini alır.
2. Eski notebook'un `data/raw/versions/` klasöründeki pre-commit snapshot'larını tarar.
3. Boş ve aynı içeriğe sahip tekrar durumlarını atlar.
4. Tarihsel benzersiz veri durumlarını sırasıyla `raw_v001.json`, `raw_v002.json`, ... yapar.
5. En son mevcut master durumunu da snapshot olarak kaydeder.
6. Her raw versiyon için manifest ve SHA-256 oluşturur.

Örneğin eski notebook ile iki 60'lık batch işlendi ve eski snapshot'lar hâlâ mevcutsa
tipik olarak:

```text
raw_v001.json → ilk batch sonrası 60 soru
raw_v002.json → iki batch sonrası 120 soru (mevcut master)
```

oluşur.

Eski snapshot geçmişi eksikse bile **mevcut master mutlaka korunur ve snapshot'a alınır**.

In [ ]:
import hashlib
import json
import shutil
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path


def canonical_dataset_bytes(dataset) -> bytes:
    """Formatting/key-order differences independent content representation."""
    payload = json.dumps(
        dataset,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )
    return payload.encode("utf-8")


def canonical_dataset_sha256(dataset) -> str:
    return hashlib.sha256(
        canonical_dataset_bytes(dataset)
    ).hexdigest()


def file_sha256(path: Path) -> str:
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


def load_json_list(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(
            f"Dataset root list değil: {path}"
        )

    return data


def get_existing_raw_versions():
    versions = []

    for path in RAW_SNAPSHOTS_DIR.glob("raw_v*.json"):
        match = re.fullmatch(
            r"raw_v(\d{3})\.json",
            path.name,
        )

        if match:
            versions.append(
                (int(match.group(1)), path)
            )

    return sorted(versions)


def next_raw_version_number() -> int:
    existing = get_existing_raw_versions()

    if not existing:
        return 1

    return max(v for v, _ in existing) + 1


def existing_snapshot_content_hashes():
    result = {}

    for version_number, path in get_existing_raw_versions():
        data = load_json_list(path)
        result[
            canonical_dataset_sha256(data)
        ] = {
            "version_number": version_number,
            "path": path,
            "n_questions": len(data),
        }

    return result


def create_raw_snapshot(
    dataset,
    *,
    origin: str,
    source_path: str = "",
    note: str = "",
):
    """
    Immutable snapshot oluşturur.
    Aynı dataset içeriği daha önce snapshot'landıysa yeni dosya oluşturmaz.
    """
    if not isinstance(dataset, list):
        raise TypeError("dataset list olmalıdır.")

    if len(dataset) == 0:
        print("Boş dataset snapshot'lanmadı.")
        return None

    content_hash = canonical_dataset_sha256(dataset)
    existing_hashes = existing_snapshot_content_hashes()

    if content_hash in existing_hashes:
        existing = existing_hashes[content_hash]
        print(
            "Aynı içerik zaten snapshot olarak mevcut:",
            existing["path"].name,
            f"({existing['n_questions']} soru)",
        )
        return existing["path"]

    version_number = next_raw_version_number()
    version_name = f"raw_v{version_number:03d}"

    snapshot_path = (
        RAW_SNAPSHOTS_DIR /
        f"{version_name}.json"
    )

    manifest_path = (
        RAW_MANIFESTS_DIR /
        f"{version_name}_manifest.json"
    )

    if snapshot_path.exists():
        raise FileExistsError(
            f"Immutable snapshot üzerine yazma engellendi: {snapshot_path}"
        )

    with open(snapshot_path, "x", encoding="utf-8") as f:
        json.dump(
            dataset,
            f,
            ensure_ascii=False,
            indent=2,
        )

    manifest = {
        "raw_dataset_version": version_name,
        "dataset_file": snapshot_path.name,
        "n_questions": len(dataset),
        "created_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "origin": origin,
        "source_path": source_path,
        "note": note,
        "file_sha256": file_sha256(snapshot_path),
        "canonical_content_sha256": content_hash,
        "answer_type_distribution": dict(
            Counter(
                item.get("answer_type", "")
                for item in dataset
            )
        ),
        "category_distribution": dict(
            Counter(
                item.get("category", "")
                for item in dataset
            )
        ),
    }

    with open(manifest_path, "x", encoding="utf-8") as f:
        json.dump(
            manifest,
            f,
            ensure_ascii=False,
            indent=2,
        )

    print(
        f"Oluşturuldu: {snapshot_path.name} "
        f"({len(dataset)} soru)"
    )

    return snapshot_path


def create_safety_backup_of_master(
    reason: str,
):
    if not MASTER_JSON_PATH.exists():
        return None

    timestamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S_%f"
    )

    backup_path = (
        SAFETY_BACKUPS_DIR /
        f"questions_master_{reason}_{timestamp}.json"
    )

    shutil.copy2(
        MASTER_JSON_PATH,
        backup_path,
    )

    print("Safety backup:", backup_path)
    return backup_path


def migrate_existing_history_to_raw_versions():
    """
    MASTER JSON'A YAZMA YAPMAZ.

    Eski pre-commit snapshot'larından ve mevcut master'dan
    benzersiz dataset durumlarını yeni raw_vXXX sistemine taşır.
    """
    if not MASTER_JSON_PATH.exists():
        raise FileNotFoundError(
            f"Master bulunamadı: {MASTER_JSON_PATH}"
        )

    master_before = load_master_dataset()
    master_hash_before = canonical_dataset_sha256(
        master_before
    )
    master_count_before = len(master_before)

    print("=" * 70)
    print("MIGRATION ÖNCESİ MASTER")
    print("Soru sayısı:", master_count_before)
    print("Canonical SHA-256:", master_hash_before)

    # Önce master'ın birebir güvenlik kopyası.
    create_safety_backup_of_master(
        reason="before_versioning_migration"
    )

    candidates = []

    if LEGACY_VERSIONS_DIR.exists():
        legacy_files = sorted(
            LEGACY_VERSIONS_DIR.glob(
                "questions_master_before_*.json"
            ),
            key=lambda p: p.name,
        )

        for path in legacy_files:
            try:
                data = load_json_list(path)
            except Exception as exc:
                print(
                    "UYARI: legacy snapshot okunamadı:",
                    path.name,
                    repr(exc),
                )
                continue

            if not data:
                # Batch-1 öncesindeki boş master gibi durumları atla.
                continue

            candidates.append(
                (
                    data,
                    "legacy_pre_commit_snapshot",
                    str(path),
                    "Eski 01 notebook geçmişinden migrate edildi.",
                )
            )

    # Mevcut master her zaman son adaydır.
    candidates.append(
        (
            master_before,
            "current_master_migration",
            str(MASTER_JSON_PATH),
            "Versioning sistemine geçiş anındaki mevcut living master.",
        )
    )

    # Candidate'ları kendi içinde content hash ile deduplicate et.
    unique_candidates = []
    seen = set()

    for data, origin, source_path, note in candidates:
        content_hash = canonical_dataset_sha256(data)

        if content_hash in seen:
            continue

        seen.add(content_hash)
        unique_candidates.append(
            (data, origin, source_path, note)
        )

    print(
        "Benzersiz tarihsel dataset durumu:",
        len(unique_candidates),
    )

    created_or_existing = []

    for data, origin, source_path, note in unique_candidates:
        path = create_raw_snapshot(
            data,
            origin=origin,
            source_path=source_path,
            note=note,
        )

        if path is not None:
            created_or_existing.append(path)

    # Migration sonrası master'ın değişmediğini kanıtla.
    master_after = load_master_dataset()
    master_hash_after = canonical_dataset_sha256(
        master_after
    )
    master_count_after = len(master_after)

    print("=" * 70)
    print("MIGRATION SONRASI MASTER")
    print("Soru sayısı:", master_count_after)
    print("Canonical SHA-256:", master_hash_after)

    if (
        master_count_before != master_count_after
        or master_hash_before != master_hash_after
    ):
        raise RuntimeError(
            "KRİTİK: Migration sırasında master değişti. "
            "Safety backup kullanılmalı."
        )

    print(
        "BAŞARILI: questions_master.json migration sırasında "
        "hiç değişmedi."
    )

    print("\nRAW VERSİYONLAR")
    for version_number, path in get_existing_raw_versions():
        data = load_json_list(path)
        print(
            f"raw_v{version_number:03d}: "
            f"{len(data)} soru — {path}"
        )

    return created_or_existing


# BU HÜCREYİ VERSIONING'E GEÇİŞTE ÇALIŞTIR:
MIGRATED_SNAPSHOTS = (
    migrate_existing_history_to_raw_versions()
)

## 5. Otomatik `question_id`

ID biçimi:

```text
TRQA000001
TRQA000002
...
```

Yeni batch eklerken elle ID yazılmayacaktır.

In [ ]:
QUESTION_ID_PATTERN = re.compile(r"^TRQA(\d{6})$")


def get_next_question_number(dataset):
    max_number = 0
    for item in dataset:
        question_id = item.get("question_id", "")
        match = QUESTION_ID_PATTERN.fullmatch(question_id)
        if match:
            max_number = max(max_number, int(match.group(1)))
    return max_number + 1


def make_question_id(number: int) -> str:
    return f"TRQA{number:06d}"

## 6. Tek kayıt doğrulama

Yeni kayıt eklenmeden önce zorunlu alanlar, tipler, alias listesi ve kontrollü vocabulary alanları doğrulanır.

In [ ]:
def validate_candidate(candidate: dict):
    errors = []
    warnings = []

    if not isinstance(candidate, dict):
        return ["Kayıt dict olmalıdır."], warnings

    missing = REQUIRED_INPUT_FIELDS - set(candidate.keys())
    if missing:
        errors.append(f"Eksik zorunlu alanlar: {sorted(missing)}")
        return errors, warnings

    question = candidate.get("question")
    reference_answer = candidate.get("reference_answer")
    aliases = candidate.get("accepted_aliases")
    answer_type = candidate.get("answer_type")
    category = candidate.get("category")

    if not isinstance(question, str) or not question.strip():
        errors.append("question boş olmayan string olmalıdır.")

    if not isinstance(reference_answer, str) or not reference_answer.strip():
        errors.append("reference_answer boş olmayan string olmalıdır.")

    if not isinstance(category, str) or not category.strip():
        errors.append("category boş olmayan string olmalıdır.")

    if answer_type not in ANSWER_TYPES:
        errors.append(
            f"Geçersiz answer_type={answer_type!r}. "
            f"İzin verilenler: {sorted(ANSWER_TYPES)}"
        )

    if not isinstance(aliases, list) or not aliases:
        errors.append("accepted_aliases boş olmayan list olmalıdır.")
    else:
        bad_aliases = [
            alias for alias in aliases
            if not isinstance(alias, str) or not alias.strip()
        ]
        if bad_aliases:
            errors.append("accepted_aliases içinde boş/geçersiz değer var.")

        normalized_aliases = [
            normalize_alias_for_duplicate_check(alias)
            for alias in aliases
            if isinstance(alias, str) and alias.strip()
        ]

        if len(normalized_aliases) != len(set(normalized_aliases)):
            warnings.append("accepted_aliases içinde duplicate alias var.")

        canonical_norm = normalize_alias_for_duplicate_check(
            reference_answer if isinstance(reference_answer, str) else ""
        )
        if canonical_norm and canonical_norm not in set(normalized_aliases):
            warnings.append(
                "reference_answer accepted_aliases içinde yok; "
                "ekleme sırasında otomatik eklenecek."
            )

    source_type = candidate.get("source_type", "generated")
    if source_type not in SOURCE_TYPES:
        errors.append(
            f"Geçersiz source_type={source_type!r}. "
            f"İzin verilenler: {sorted(SOURCE_TYPES)}"
        )

    verification_status = candidate.get("verification_status", "unverified")
    if verification_status not in VERIFICATION_STATUSES:
        errors.append(
            f"Geçersiz verification_status={verification_status!r}. "
            f"İzin verilenler: {sorted(VERIFICATION_STATUSES)}"
        )

    return errors, warnings

## 7. Batch hazırlama ve dry-run

Yeni sorular önce veri tabanına yazılmadan kontrol edilir. Hata yoksa ayrı bir adımda commit edilir.

In [ ]:
from copy import deepcopy
from datetime import datetime, timezone


def prepare_candidate(candidate: dict, batch_id: str) -> dict:
    item = deepcopy(candidate)

    for key, default_value in OPTIONAL_INPUT_DEFAULTS.items():
        item.setdefault(key, default_value)

    item["question"] = item["question"].strip()
    item["reference_answer"] = item["reference_answer"].strip()
    item["category"] = item["category"].strip()

    aliases = [
        alias.strip()
        for alias in item["accepted_aliases"]
        if isinstance(alias, str) and alias.strip()
    ]

    canonical = item["reference_answer"]
    canonical_norm = normalize_alias_for_duplicate_check(canonical)

    if canonical_norm not in {
        normalize_alias_for_duplicate_check(alias)
        for alias in aliases
    }:
        aliases.insert(0, canonical)

    unique_aliases = []
    seen_aliases = set()
    for alias in aliases:
        key = normalize_alias_for_duplicate_check(alias)
        if key not in seen_aliases:
            seen_aliases.add(key)
            unique_aliases.append(alias)

    item["accepted_aliases"] = unique_aliases
    item["batch_id"] = batch_id
    return item


def inspect_batch(new_batch, master_dataset):
    existing_questions = {
        normalize_for_duplicate_check(item["question"]): item["question_id"]
        for item in master_dataset
        if item.get("question")
    }

    batch_seen = {}
    report = []
    prepared_items = []
    batch_id = datetime.now().strftime("batch_%Y%m%d_%H%M%S")

    for index, candidate in enumerate(new_batch, start=1):
        errors, warnings = validate_candidate(candidate)

        if errors:
            report.append({
                "row": index,
                "status": "ERROR",
                "errors": errors,
                "warnings": warnings,
                "question": candidate.get("question", "")
                if isinstance(candidate, dict) else "",
            })
            continue

        prepared = prepare_candidate(candidate, batch_id=batch_id)
        normalized_question = normalize_for_duplicate_check(prepared["question"])

        if normalized_question in existing_questions:
            errors.append(
                "Master dataset içinde duplicate soru: "
                f"{existing_questions[normalized_question]}"
            )

        if normalized_question in batch_seen:
            errors.append(
                "Aynı yeni batch içinde duplicate soru: "
                f"satır {batch_seen[normalized_question]}"
            )
        else:
            batch_seen[normalized_question] = index

        status = "ERROR" if errors else ("WARNING" if warnings else "OK")
        report.append({
            "row": index,
            "status": status,
            "errors": errors,
            "warnings": warnings,
            "question": prepared["question"],
        })

        if not errors:
            prepared_items.append(prepared)

    return {
        "batch_id": batch_id,
        "report": report,
        "prepared_items": prepared_items,
        "has_errors": any(r["status"] == "ERROR" for r in report),
    }

## 8. Yeni batch alanı

Bundan sonra benden istediğin soru batch'lerini bu şemada vereceğim. Sen yalnızca `NEW_BATCH` listesinin içine yapıştıracaksın.

Örnek kayıt:

```python
{
    "question": "Türkiye'nin başkenti neresidir?",
    "reference_answer": "Ankara",
    "accepted_aliases": ["Ankara"],
    "answer_type": "ENTITY_LOCATION",
    "category": "Coğrafya",
    "source_type": "generated",
    "source_name": "",
    "source_reference": "",
    "verification_status": "unverified",
    "notes": ""
}
```

In [ ]:
NEW_BATCH = [
    # Buraya yeni soru kayıtları gelecek.
]

print("Yeni batch boyutu:", len(NEW_BATCH))

## 9. Dry-run raporu

In [ ]:
import pandas as pd

master_data = load_master_dataset()
inspection = inspect_batch(NEW_BATCH, master_data)
report_df = pd.DataFrame(inspection["report"])

print("Batch ID:", inspection["batch_id"])
print("Hata var mı?:", inspection["has_errors"])
print("Yazılabilir kayıt:", len(inspection["prepared_items"]))

display(report_df)

## 10. Batch'i master JSON'a commit et + otomatik yeni raw snapshot

Dry-run raporu temizse bu aşama:

1. Commit öncesi mevcut master'ın safety backup'ını alır.
2. Yeni kayıtları yaşayan `questions_master.json` dosyasına ekler.
3. ID bütünlüğünü doğrular.
4. Başarılı commit sonrasında master'ın **yeni halini** `raw_vXXX.json` olarak snapshot'lar.
5. Snapshot'ın manifest ve checksum'unu üretir.

### Güvenlik

- Önceki `raw_vXXX` dosyalarının üzerine yazılmaz.
- Aynı dataset hali tekrar snapshot'lanmaz.
- Eski iki batch'iniz silinmez veya yeniden numaralandırılmaz.
- `NEW_BATCH` içinde daha önce commit edilmiş soruları tekrar bırakmayın;
  dry-run duplicate kontrolü zaten bunu engeller.

In [ ]:
import shutil


def append_inspected_batch(inspection_result):
    if inspection_result["has_errors"]:
        raise ValueError(
            "Batch içinde ERROR var. Master dataset değiştirilmedi."
        )

    items_to_add = deepcopy(
        inspection_result["prepared_items"]
    )

    if not items_to_add:
        print("Eklenecek kayıt yok.")
        return []

    master_dataset = load_master_dataset()

    # Commit anındaki güncel master'a karşı duplicate kontrolü.
    second_check = inspect_batch(
        items_to_add,
        master_dataset,
    )

    if second_check["has_errors"]:
        raise ValueError(
            "Master veri değişmiş veya duplicate oluşmuş. "
            "Batch yeniden dry-run edilmelidir."
        )

    master_count_before = len(master_dataset)
    master_hash_before = canonical_dataset_sha256(
        master_dataset
    )

    # Geri dönüş için pre-commit safety backup.
    create_safety_backup_of_master(
        reason="before_batch_commit"
    )

    next_number = get_next_question_number(
        master_dataset
    )
    added_at = datetime.now(
        timezone.utc
    ).isoformat()

    added_records = []

    for offset, item in enumerate(items_to_add):
        record = deepcopy(item)

        record["question_id"] = make_question_id(
            next_number + offset
        )

        record["created_at_utc"] = added_at

        ordered_record = {
            "question_id": record["question_id"],
            "question": record["question"],
            "reference_answer": record["reference_answer"],
            "accepted_aliases": record["accepted_aliases"],
            "answer_type": record["answer_type"],
            "category": record["category"],
            "source_type": record["source_type"],
            "source_name": record["source_name"],
            "source_reference": record["source_reference"],
            "verification_status": record["verification_status"],
            "notes": record["notes"],
            "batch_id": record["batch_id"],
            "created_at_utc": record["created_at_utc"],
        }

        master_dataset.append(
            ordered_record
        )

        added_records.append(
            ordered_record
        )

    # Living master'ı güncelle.
    save_master_dataset(
        master_dataset
    )

    reloaded = load_master_dataset()

    # Yazma sonrası ID bütünlüğü.
    ids = [
        item["question_id"]
        for item in reloaded
    ]

    if len(ids) != len(set(ids)):
        raise RuntimeError(
            "KRİTİK: Yazma sonrası duplicate "
            "question_id bulundu."
        )

    expected_count = (
        master_count_before
        + len(added_records)
    )

    if len(reloaded) != expected_count:
        raise RuntimeError(
            "KRİTİK: Commit sonrası beklenmeyen "
            "soru sayısı."
        )

    print(
        f"{len(added_records)} kayıt başarıyla eklendi."
    )

    print(
        "Master soru sayısı:",
        f"{master_count_before} -> {len(reloaded)}"
    )

    # Başarılı commit SONRASINDA immutable snapshot.
    snapshot_path = create_raw_snapshot(
        reloaded,
        origin="post_batch_commit",
        source_path=str(MASTER_JSON_PATH),
        note=(
            "Başarılı batch commit sonrası otomatik "
            "raw dataset snapshot."
        ),
    )

    print(
        "Commit öncesi master hash:",
        master_hash_before,
    )

    print(
        "Commit sonrası master hash:",
        canonical_dataset_sha256(reloaded),
    )

    if snapshot_path is not None:
        print(
            "Aktif raw snapshot:",
            snapshot_path.name,
        )

    return added_records


# Dry-run raporunu kontrol ettikten sonra çalıştır:
# ADDED_RECORDS = append_inspected_batch(inspection)

## 10B. Mevcut raw dataset versiyonlarını listele

Bu hücre living master'a dokunmaz.
Hangi immutable versiyonların mevcut olduğunu ve soru sayılarını gösterir.

In [ ]:
raw_version_rows = []

for version_number, path in get_existing_raw_versions():
    data = load_json_list(path)

    manifest_path = (
        RAW_MANIFESTS_DIR /
        f"raw_v{version_number:03d}_manifest.json"
    )

    raw_version_rows.append({
        "raw_version": f"raw_v{version_number:03d}",
        "n_questions": len(data),
        "path": str(path),
        "manifest_exists": manifest_path.exists(),
        "canonical_sha256": canonical_dataset_sha256(data),
    })

raw_versions_df = pd.DataFrame(
    raw_version_rows
)

display(raw_versions_df)

## 11. Dataset dağılım özeti

Her batch sonrasında answer type, kategori, verification status ve source type dağılımları kontrol edilir.

In [ ]:
from collections import Counter


def dataset_summary(dataset):
    if not dataset:
        print("Dataset boş.")
        return

    groups = {
        "ANSWER TYPE": Counter(item["answer_type"] for item in dataset),
        "CATEGORY": Counter(item["category"] for item in dataset),
        "VERIFICATION STATUS": Counter(item.get("verification_status", "") for item in dataset),
        "SOURCE TYPE": Counter(item.get("source_type", "") for item in dataset),
    }

    print("=" * 60)
    print("MASTER DATASET ÖZETİ")
    print("=" * 60)
    print("Toplam soru:", len(dataset))

    for title, counts in groups.items():
        print(f"\n{title}")
        for key, value in counts.most_common():
            print(f"{key:30s} {value:5d} ({value / len(dataset):6.2%})")


master_data = load_master_dataset()
dataset_summary(master_data)

## 12. Detaylı tablo görünümü

In [ ]:
master_data = load_master_dataset()

if master_data:
    master_df = pd.DataFrame(master_data)
    display(
        master_df[[
            "question_id",
            "question",
            "reference_answer",
            "answer_type",
            "category",
            "verification_status",
            "batch_id",
        ]].tail(50)
    )
else:
    print("Master dataset henüz boş.")

## 13. Master dataset bütünlük kontrolü

Bu hızlı kontrol yapısal bozulmaları erken yakalamak içindir. Daha ayrıntılı içerik ve bilimsel kalite kontrolü `02_dataset_validation.ipynb` içinde yapılacaktır.

In [ ]:
def validate_master_integrity(dataset):
    problems = []
    ids = []
    normalized_questions = {}

    required_saved_fields = {
        "question_id", "question", "reference_answer", "accepted_aliases",
        "answer_type", "category", "source_type", "source_name",
        "source_reference", "verification_status", "notes", "batch_id",
        "created_at_utc",
    }

    for row_index, item in enumerate(dataset, start=1):
        question_id = item.get("question_id", "")
        ids.append(question_id)

        if not QUESTION_ID_PATTERN.fullmatch(question_id):
            problems.append(
                f"Satır {row_index}: geçersiz question_id={question_id!r}"
            )

        missing = required_saved_fields - set(item.keys())
        if missing:
            problems.append(f"{question_id}: eksik alanlar {sorted(missing)}")

        candidate_without_internal_fields = {
            k: v for k, v in item.items()
            if k not in {"question_id", "batch_id", "created_at_utc"}
        }
        errors, _ = validate_candidate(candidate_without_internal_fields)
        for error in errors:
            problems.append(f"{question_id}: {error}")

        normalized_question = normalize_for_duplicate_check(
            item.get("question", "")
        )
        if normalized_question in normalized_questions:
            problems.append(
                f"Duplicate soru: {question_id} ve "
                f"{normalized_questions[normalized_question]}"
            )
        else:
            normalized_questions[normalized_question] = question_id

    if len(ids) != len(set(ids)):
        problems.append("Duplicate question_id mevcut.")

    if problems:
        print("BÜTÜNLÜK KONTROLÜ BAŞARISIZ")
        for problem in problems:
            print("-", problem)
    else:
        print("BÜTÜNLÜK KONTROLÜ BAŞARILI")
        print("Toplam soru:", len(dataset))

    return problems


master_data = load_master_dataset()
integrity_problems = validate_master_integrity(master_data)

# Notebook 01 — çalışma kuralları

Bu notebook veri üretimi devam ettiği sürece **yaşayan bir notebook** olarak kullanılacaktır.

Her yeni batch için:

- [ ] `NEW_BATCH` yalnızca yeni soruları içeriyor
- [ ] Dry-run sonucu `ERROR=False`
- [ ] Commit öncesi safety backup oluşuyor
- [ ] Master soru sayısı beklenen kadar artıyor
- [ ] Başarılı commit sonrası yeni `raw_vXXX.json` oluşuyor
- [ ] Eski raw versiyonlar değişmeden duruyor
- [ ] Yeni raw manifest ve checksum oluşuyor
- [ ] Master bütünlük kontrolü hatasız

## Dataset'i deneylerde kullanma

Model/feature notebook'ları `questions_master.json` okumamalıdır.

Önce belirli bir raw sürüm seçilir:

```text
raw_v001
raw_v002
raw_v003
...
```

Bu sürüm `02_dataset_validation.ipynb` ile validate edilir.
Sonraki deneyler de seçilmiş `validated_vXXX` üzerinde yürütülür.

Veri üretimine paralel olarak yeni raw sürümler oluşturmaya devam edebilirsiniz.